# 📊 Análisis de Rendimiento — Modelos de IA
**Telco Customer Churn | ITY1101 — Evaluación Parcial N°3**

Métricas evaluadas: Matriz de Confusión, Accuracy, Recall, Precisión, F1 Score, Curva ROC, AUC y Gini

## 📦 1. Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score,
    classification_report, ConfusionMatrixDisplay
)

print('✅ Librerías importadas correctamente')

## 📥 2. Cargar y Preparar Datos

In [ ]:
# Cargar dataset limpio del pipeline DataOps
df = pd.read_csv('IA_Proyecto/data/telco_limpio.csv')
print(f'✅ Dataset cargado: {df.shape[0]} filas | {df.shape[1]} columnas')

# Codificar variables categóricas
df_ml = df.copy()
le = LabelEncoder()
for col in df_ml.select_dtypes(include=['object']).columns:
    df_ml[col] = le.fit_transform(df_ml[col])

# Separar features y variable objetivo
X = df_ml.drop('Churn', axis=1)
y = df_ml['Churn']

# División 80% entrenamiento / 20% prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'✅ Entrenamiento: {len(X_train)} registros ({len(X_train)/len(X)*100:.0f}%)')
print(f'✅ Prueba       : {len(X_test)} registros ({len(X_test)/len(X)*100:.0f}%)')
print(f'✅ Distribución Churn en prueba: {y_test.value_counts().to_dict()}')

## 🤖 3. Entrenamiento de los 3 Modelos

In [ ]:
# Escalar datos para Regresión Logística
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modelo 1: Regresión Logística
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

# Modelo 2: Árbol de Decisión  
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

# Modelo 3: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Predicciones
y_pred_lr = lr.predict(X_test_scaled)
y_pred_dt = dt.predict(X_test)
y_pred_rf = rf.predict(X_test)

# Probabilidades para ROC
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]
y_prob_dt = dt.predict_proba(X_test)[:, 1]
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print('✅ Los 3 modelos entrenados correctamente')

## 📊 4. Métricas de Rendimiento

In [ ]:
def calcular_metricas(nombre, y_test, y_pred, y_prob):
    acc  = accuracy_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)
    gini = 2 * auc - 1
    return {'Modelo': nombre, 'Accuracy': acc, 'Recall': rec,
            'Precisión': prec, 'F1 Score': f1, 'AUC': auc, 'Gini': gini}

resultados = pd.DataFrame([
    calcular_metricas('Regresión Logística', y_test, y_pred_lr, y_prob_lr),
    calcular_metricas('Árbol de Decisión',   y_test, y_pred_dt, y_prob_dt),
    calcular_metricas('Random Forest',       y_test, y_pred_rf, y_prob_rf),
])

print('=' * 70)
print('TABLA COMPARATIVA DE MÉTRICAS')
print('=' * 70)
cols_num = ['Accuracy','Recall','Precisión','F1 Score','AUC','Gini']
display_df = resultados.copy()
for c in cols_num:
    display_df[c] = display_df[c].map(lambda x: f'{x:.4f}')
print(display_df.to_string(index=False))

mejor = resultados.loc[resultados['F1 Score'].idxmax()]
print(f'\n🏆 Mejor modelo por F1 Score: {mejor["Modelo"]} ({mejor["F1 Score"]:.4f})')

## 🔲 5. Matrices de Confusión

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

modelos = [
    ('Regresión Logística', y_pred_lr),
    ('Árbol de Decisión', y_pred_dt),
    ('Random Forest', y_pred_rf)
]

for ax, (nombre, y_pred) in zip(axes, modelos):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{nombre}\nAccuracy: {accuracy_score(y_test, y_pred):.2%}', fontweight='bold')

plt.suptitle('Matrices de Confusión — Comparación de Modelos', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('IA_Proyecto/data/matrices_confusion.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Matrices de confusión generadas')

## 📈 6. Curva ROC y Coeficiente Gini

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Curva ROC ──────────────────────────────────────────────────────────────────
ax1 = axes[0]
colores = ['#3498db', '#e67e22', '#2ecc71']
nombres = ['Regresión Logística', 'Árbol de Decisión', 'Random Forest']
probs = [y_prob_lr, y_prob_dt, y_prob_rf]

for nombre, prob, color in zip(nombres, probs, colores):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    gini = 2 * auc - 1
    ax1.plot(fpr, tpr, color=color, linewidth=2,
             label=f'{nombre} (AUC={auc:.3f}, Gini={gini:.3f})')

ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Modelo aleatorio')
ax1.set_xlabel('Tasa de Falsos Positivos (FPR)')
ax1.set_ylabel('Tasa de Verdaderos Positivos (TPR)')
ax1.set_title('Curva ROC — Comparación de Modelos', fontweight='bold')
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(alpha=0.3)

# ── Gini ───────────────────────────────────────────────────────────────────────
ax2 = axes[1]
ginis = [2*roc_auc_score(y_test, p)-1 for p in probs]
bars = ax2.bar(nombres, ginis, color=colores, alpha=0.85, edgecolor='white')
ax2.set_title('Coeficiente de Gini por Modelo', fontweight='bold')
ax2.set_ylabel('Gini')
ax2.set_ylim(0, 1)
ax2.axhline(0.6, color='red', linestyle='--', alpha=0.7, label='Umbral buen modelo (0.6)')
ax2.legend()

for bar, val in zip(bars, ginis):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('IA_Proyecto/data/roc_gini.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Curva ROC y Gini generados')

## 📋 7. Reporte Detallado del Mejor Modelo

In [ ]:
print('=' * 60)
print('  REPORTE DETALLADO — RANDOM FOREST (MEJOR MODELO)')
print('=' * 60)
print(classification_report(y_test, y_pred_rf, target_names=['No Churn', 'Churn']))

auc_rf = roc_auc_score(y_test, y_prob_rf)
gini_rf = 2 * auc_rf - 1

print(f'AUC    : {auc_rf:.4f}')
print(f'Gini   : {gini_rf:.4f}')
print()
print('INTERPRETACIÓN:')
print(f'  - El modelo acierta el {accuracy_score(y_test, y_pred_rf):.1%} de las predicciones')
print(f'  - Detecta el {recall_score(y_test, y_pred_rf):.1%} de los clientes que harán Churn')
print(f'  - Gini de {gini_rf:.3f} indica un modelo con buena capacidad discriminatoria')
print('=' * 60)

## 🏆 8. Conclusión Final

In [ ]:
print('=' * 60)
print('  RESUMEN COMPARATIVO FINAL')
print('=' * 60)
for _, row in resultados.iterrows():
    print(f"\n{row['Modelo']}:")
    print(f"  Accuracy : {row['Accuracy']:.2%}")
    print(f"  Recall   : {row['Recall']:.2%}")
    print(f"  F1 Score : {row['F1 Score']:.4f}")
    print(f"  Gini     : {row['Gini']:.4f}")

mejor = resultados.loc[resultados['F1 Score'].idxmax()]
print(f"\n{'='*60}")
print(f"  🏆 MEJOR MODELO: {mejor['Modelo']}")
print(f"  Justificación: Mayor F1 Score ({mejor['F1 Score']:.4f}) que")
print(f"  equilibra precisión y recall en dataset desbalanceado")
print(f"  (solo 26.5% de clientes hace Churn)")
print('=' * 60)